In [1]:

from pathlib import Path
from tqdm import tqdm

from samples.sample import Sample

📂 2. Define dataset path

In [2]:
DATA_DIR = Path("/flux/vault/stator/03_splitting/stator_splitting_ds2_nc/train/")

# List all .h5 sample files
sample_files = list(DATA_DIR.glob("*.h5"))
len(sample_files)


611

🔍 3. Inspect a few samples manually

In [3]:
def inspect_sample(path):
    s = Sample.from_file(path)
    
    print("File:", path.name)
    print("UUID:", s.uuid)
    
    print("\nAvailable datablocks:")
    print(s.list_data_types())
    
    print("\nFlags:")
    flags = s.get("flags")
    if flags:
        print(flags)
    else:
        print("No flags found")

# Try a few samples
for p in sample_files[:3]:
    inspect_sample(p)
    print("="*50)


File: sample_536.h5
UUID: 92e1d41f-1b36-43b1-bd62-73e8cac0d0a5

Available datablocks:
{'flags': 'scalar_data', 'heatflux_matrix': 'scalar_data', 'stator_scalars': 'scalar_data'}

Flags:
File: sample_879.h5
UUID: b4464e97-4ad5-40ff-b8c6-5630c6a97efa

Available datablocks:
{'flags': 'scalar_data', 'heatflux_matrix': 'scalar_data', 'stator_scalars': 'scalar_data'}

Flags:
File: sample_454.h5
UUID: 931219ef-a614-45f7-8fb9-7c3b6d144108

Available datablocks:
{'flags': 'scalar_data', 'heatflux_matrix': 'scalar_data', 'stator_scalars': 'scalar_data'}

Flags:


✅ 4. Define validation logic
We want to remove samples if:

❌ Condition 1:

heatflux_matrix does NOT exist

❌ Condition 2:

flags["missing_heatflux"] == True

In [61]:
def is_valid_sample(path):
    try:
        s = Sample.from_file(path)
        
        # --- Check 1: heatflux_matrix exists ---
        if 'heatflux_matrix' not in (s.to_dict()['metadata']['datablocks']).keys():
            return False
        
        # --- Check 2: flags ---
        flags_block = s.get("flags")
        
        if flags_block is not None:
            # Assuming flags behaves like dict or has .data
            flags = flags_block.read()
            
            # adjust if needed:
            if "missing_heatflux" in flags and flags.get("missing_heatflux") == True:
                return False
         
        return True

    except Exception as e:
        print(f"Error reading {path}: {e}")
        return False


In [39]:
s.keys()

<KeysViewHDF5 ['data']>

In [57]:
flags_block = s.get("flags")
if flags_block is not None:
    # Assuming flags behaves like dict or has .data
    flags = flags_block.read()
    print(flags)
    print("yes")
    # adjust if needed:
    if "missing_heatflux" in flags and flags.get("missing_heatflux") == False:
        print(flags)


{'high_a_matrix_error': array([False]), 'high_q_error': array([False]), 'missing_heatflux': array([False])}
yes
{'high_a_matrix_error': array([False]), 'high_q_error': array([False]), 'missing_heatflux': array([False])}


In [6]:
for p in tqdm(sample_files):
    print(p)
    break

  0%|          | 0/800 [00:00<?, ?it/s]

/flux/vault/stator/03_splitting/stator_splitting_ds2_nc/train/sample_536.h5


📊 5. Analyse dataset (before deletion)

In [62]:

valid_samples = []
invalid_samples = []

for p in tqdm(sample_files):
    if is_valid_sample(p):
        valid_samples.append(p)
    else:
        invalid_samples.append(p)

print("Total samples:", len(sample_files))
print("Valid samples:", len(valid_samples))
print("Invalid samples:", len(invalid_samples))

100%|██████████| 800/800 [00:35<00:00, 22.71it/s]

Total samples: 800
Valid samples: 608
Invalid samples: 192
